# Train the railway door-inspection detector (YOLOE-26s-seg)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProfessorYaseen/railway-door-inspection-robot/blob/main/training/train_door_detector.ipynb)

Fine-tunes **YOLOE-26s-seg** on the four inspection classes and exports it for the robot's Intel NUC.

| Class | Meaning |
|---|---|
| `Door_Open` | door fully open |
| `Door_closed` | door correctly closed |
| `Door_not_closed` | door ajar / not latched (safety-critical) |
| `Rubber_missing` | missing door-seal rubber (maintenance defect) |

**Runtime:** GPU (Tesla T4 is enough, ~36 min for 100 epochs).

**Data:** the dataset is private. Upload your Roboflow export (*YOLOv8 / YOLO11* format) as `dataset_clean.zip`.

In [ ]:
!pip install -q ultralytics
!git clone -q https://github.com/ProfessorYaseen/railway-door-inspection-robot.git
%cd railway-door-inspection-robot

## 1. Unzip the dataset
Expected layout after unzipping: `data/data.yaml`, `data/{train,valid,test}/{images,labels}`.

In [ ]:
import zipfile
zipfile.ZipFile("/content/dataset_clean.zip").extractall("data")
!ls data

## 2. Prepare
* rewrites `data.yaml` paths so they work on any machine;
* converts **box** labels to 4-corner **polygons** – the YOLOE segmentation trainer
  refuses a dataset that mixes boxes and polygons;
* prints per-split image counts (empty-label images are deliberate *hard negatives*).

In [ ]:
!python training/train.py prepare --data data/data.yaml

## 3. Train
AdamW, lr 1e-3, weight decay 0.025, 5 warm-up epochs, mosaic + 0.1 mixup (mosaic off for the last 10 epochs), early-stopping patience 20.

In [ ]:
!python training/train.py train --data data/data.yaml --epochs 100 --batch 8 --device 0

## 4. Evaluate on the held-out **test** split
Validation metrics guide training; the test split is only looked at once, here.

In [ ]:
W = "runs/segment/train_door_detector/yoloe26s/weights/best.pt"
!python training/train.py eval --data data/data.yaml --weights {W} --split test

## 5. Export for deployment
* **OpenVINO FP16** – fastest on the robot's Intel NUC CPU
* **ONNX (opset 17)** – portable (laptop, ONNX Runtime, TensorRT)

In [ ]:
!python training/train.py export --weights {W}

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("train_door_final", "zip", "runs/segment/train_door_detector/yoloe26s")
files.download("train_door_final.zip")